# Volume Creation

we first create a volume where we are going to store all files we are going to use for this practice

In [0]:
##DBFS is disabled for Databricks Free Edition 2024+
##Unity Catalog is enabled on the other hand
##Volumes are the only way to store real files

# Create volume first
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.streaming_demo")

# Then create directory
dbutils.fs.mkdirs("/Volumes/workspace/default/streaming_demo/input")

# Stream DataFrame Creation

We define cloudFiles as the format for readStream, since the ingestion will be continous

In [0]:
##Creation of Streaming DataFrame
df_stream = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", "/Volumes/workspace/default/streaming_demo/schema")
    .load("/Volumes/workspace/default/streaming_demo/input")
)

# File loading

A JSON file is going to be added so when we have to write the stream, the schema can be inferred from this document

In [0]:
dbutils.fs.put(
    "/Volumes/workspace/default/streaming_demo/input/file1.json",
    """{"id": 1, "value": "hola"}""",
    True
)

Wrote 26 bytes.


True

Otherwise we can use spark sql types to initialize the schema without any new file

In [0]:
from pyspark.sql.types import *

schema = StructType([
    StructField("id", IntegerType()),
    StructField("value", StringType())
])


# Stream writing to Delta

Now we are going to initiate a streaming sink, in other words, we are telling Spark where to write the data arriving from the streaming.

The format will be delta, so each micro-batch that arrives becomes Delta Files. This operation will update the `_delta_log`.

Finally we will have an **ACID** table.

In the `option("checkpointLocation","Volumes/ ... /chk")`we are defining a checkpoint that will store processed offsets, the state of the stream, commits and metadata. Without this stream there is no fault tolerance.

In [0]:
#Stream writing to Delta
(df_stream.writeStream
    .format("delta")
    .option("checkpointLocation", "/Volumes/workspace/default/streaming_demo/chk")
    .option("cloudFiles.schemaLocation", "/Volumes/workspace/default/streaming_demo/schema")
    .trigger(availableNow=True)
    .outputMode("append")
    .table("workspace.default.streaming_demo"))

We add another document to validate

In [0]:
#File insertion to the stream
dbutils.fs.put("/Volumes/workspace/default/streaming_demo/input/file2.json",
               """{"id": 2, "value": "nuevo archivo 2"}""", True)

dbutils.fs.put("/Volumes/workspace/default/streaming_demo/input/file3.json",
               """{"id": 3, "value": "nuevo archivo 3"}""", True)

dbutils.fs.put("/Volumes/workspace/default/streaming_demo/input/file4.json",
               """{"id": 4, "value": "nuevo archivo 4"}""", True)

dbutils.fs.put("/Volumes/workspace/default/streaming_demo/input/file5.json",
               """{"id": 5, "value": "nuevo archivo 5"}""", True)

Wrote 37 bytes.
Wrote 37 bytes.
Wrote 37 bytes.
Wrote 37 bytes.


True

# Stream verification

We have to verify that the table is updating by itself.

If you have uploaded new files to the stream you have to run the writeStream command above to update the table (just the writeStream), the Delta table will update automatically

In [0]:
%sql
SELECT * FROM workspace.default.streaming_demo;

id,value,_rescued_data
2,nuevo archivo 2,null
3,nuevo archivo 3,null
4,nuevo archivo 4,null
5,nuevo archivo 5,null
1,hola,null


Finally we use `DESCRIBE DETAIL`to corroborate whether the delta tbale was registered in the Unity Catalog or not. An empty output of this command or a `TABLE NOT FOUND` message would mean the table was not registered.

In [0]:
%sql
DESCRIBE DETAIL workspace.default.streaming_demo;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,b14adfdf-6b54-4390-ac46-4da783e4f414,workspace.default.streaming_demo,null,,2026-04-16T04:39:29.704Z,2026-04-16T04:55:41.000Z,List(),List(),2,1997,"Map(delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true, delta.enableRowTracking -> true, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-9896634b-5d68-4c86-9874-ecf498d6148a, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-d81e1996-d0d1-45be-9803-e67b87a529c9)",3,7,"List(appendOnly, deletionVectors, domainMetadata, invariants, rowTracking)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false
